In [0]:
# PROJECT — DoorDash-Style Pricing & Revenue Strategy
# Synthetic dataset for Executive Margin Protection Dashboard

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

n = 30000

regions = {
    "West": ["Los Angeles", "San Francisco", "Seattle", "Phoenix"],
    "Northeast": ["New York", "Boston", "Philadelphia", "Washington DC"],
    "South": ["Atlanta", "Miami", "Dallas", "Charlotte"],
    "Midwest": ["Chicago", "Detroit", "Minneapolis", "Columbus"]
}

segments = ["Promo Hunter", "Casual User", "Loyal Subscriber", "Corporate Lunch", "High-Value Family"]
channels = ["Paid Social", "Referral", "Organic Search", "Email", "Partnership"]
categories = ["Fast Food", "Casual Dining", "Premium Dining", "Grocery", "Convenience"]
promo_types = ["None", "First Order Promo", "Free Delivery", "Percent Discount", "Winback Promo"]
refund_reasons = ["None", "Late Delivery", "Missing Item", "Cold Food", "Wrong Order"]

rows = []

start_date = datetime(2025, 1, 1)

for i in range(n):
    order_id = f"ORD-{i+1:06d}"
    customer_id = f"CUST-{np.random.randint(1, 8000):05d}"
    
    region = np.random.choice(list(regions.keys()), p=[0.32, 0.25, 0.25, 0.18])
    city = np.random.choice(regions[region])
    
    segment = np.random.choice(
        segments,
        p=[0.25, 0.30, 0.20, 0.10, 0.15]
    )
    
    acquisition_channel = np.random.choice(
        channels,
        p=[0.32, 0.18, 0.25, 0.15, 0.10]
    )
    
    restaurant_category = np.random.choice(
        categories,
        p=[0.30, 0.25, 0.15, 0.15, 0.15]
    )
    
    order_date = start_date + timedelta(days=np.random.randint(0, 365))
    
    # Base AOV logic
    base_aov = {
        "Fast Food": 22,
        "Casual Dining": 38,
        "Premium Dining": 68,
        "Grocery": 55,
        "Convenience": 28
    }[restaurant_category]
    
    if segment == "Corporate Lunch":
        base_aov *= 1.8
    elif segment == "High-Value Family":
        base_aov *= 1.5
    elif segment == "Promo Hunter":
        base_aov *= 0.9
    
    gross_order_value = max(10, np.random.normal(base_aov, base_aov * 0.25))
    
    # Subscription logic
    subscription_member = "Yes" if segment in ["Loyal Subscriber", "High-Value Family"] and np.random.rand() < 0.75 else "No"
    
    # Promo logic
    if segment == "Promo Hunter":
        promotion_type = np.random.choice(["First Order Promo", "Percent Discount", "Winback Promo"], p=[0.35, 0.45, 0.20])
        discount_pct = np.random.uniform(0.20, 0.45)
    elif segment == "Casual User":
        promotion_type = np.random.choice(promo_types, p=[0.35, 0.20, 0.20, 0.20, 0.05])
        discount_pct = np.random.uniform(0.05, 0.25) if promotion_type != "None" else 0
    elif segment == "Loyal Subscriber":
        promotion_type = np.random.choice(["None", "Free Delivery", "Percent Discount"], p=[0.65, 0.25, 0.10])
        discount_pct = np.random.uniform(0.00, 0.12) if promotion_type != "None" else 0
    elif segment == "Corporate Lunch":
        promotion_type = np.random.choice(["None", "Percent Discount"], p=[0.80, 0.20])
        discount_pct = np.random.uniform(0.00, 0.10) if promotion_type != "None" else 0
    else:
        promotion_type = np.random.choice(["None", "Free Delivery", "Percent Discount"], p=[0.55, 0.25, 0.20])
        discount_pct = np.random.uniform(0.03, 0.18) if promotion_type != "None" else 0
    
    discount_amount = gross_order_value * discount_pct
    
    # Fees
    delivery_fee = 0 if subscription_member == "Yes" else np.random.uniform(2.99, 7.99)
    service_fee = gross_order_value * np.random.uniform(0.08, 0.14)
    
    # Refund logic
    refund_probability = 0.04
    
    if region == "West":
        refund_probability += 0.03
    if segment == "Promo Hunter":
        refund_probability += 0.04
    if discount_pct > 0.25:
        refund_probability += 0.03
    
    has_refund = np.random.rand() < refund_probability
    refund_amount = gross_order_value * np.random.uniform(0.15, 0.75) if has_refund else 0
    refund_reason = np.random.choice(refund_reasons[1:]) if has_refund else "None"
    
    # Operational costs
    delivery_distance_miles = np.random.uniform(1, 9)
    delivery_time_minutes = np.random.normal(35, 10)
    
    driver_incentive_cost = np.random.uniform(3, 8)
    
    if region == "West":
        driver_incentive_cost *= 1.35
    if delivery_distance_miles > 6:
        driver_incentive_cost *= 1.25
    
    # CAC allocation
    cac_base = {
        "Paid Social": 9.5,
        "Referral": 4.0,
        "Organic Search": 2.5,
        "Email": 1.5,
        "Partnership": 5.5
    }[acquisition_channel]
    
    if segment == "Promo Hunter":
        marketing_cac_allocated = cac_base * np.random.uniform(1.2, 1.8)
    elif segment == "Loyal Subscriber":
        marketing_cac_allocated = cac_base * np.random.uniform(0.4, 0.8)
    else:
        marketing_cac_allocated = cac_base * np.random.uniform(0.8, 1.2)
    
    # Retention and LTV logic
    if segment == "Promo Hunter":
        retention_score = np.random.uniform(0.10, 0.35)
        lifetime_orders = np.random.randint(1, 5)
    elif segment == "Casual User":
        retention_score = np.random.uniform(0.25, 0.55)
        lifetime_orders = np.random.randint(2, 9)
    elif segment == "Loyal Subscriber":
        retention_score = np.random.uniform(0.70, 0.95)
        lifetime_orders = np.random.randint(12, 45)
    elif segment == "Corporate Lunch":
        retention_score = np.random.uniform(0.65, 0.90)
        lifetime_orders = np.random.randint(8, 35)
    else:
        retention_score = np.random.uniform(0.55, 0.85)
        lifetime_orders = np.random.randint(6, 25)
    
    new_vs_repeat = "Repeat" if lifetime_orders > 3 else "New"
    
    # Commission / margin
    commission_rate = {
        "Fast Food": 0.18,
        "Casual Dining": 0.22,
        "Premium Dining": 0.27,
        "Grocery": 0.15,
        "Convenience": 0.20
    }[restaurant_category]
    
    restaurant_commission = gross_order_value * commission_rate
    
    net_revenue = gross_order_value + delivery_fee + service_fee - discount_amount - refund_amount
    
    gross_margin = restaurant_commission + delivery_fee + service_fee - discount_amount - refund_amount
    
    contribution_profit = gross_margin - driver_incentive_cost - marketing_cac_allocated
    
    contribution_margin_pct = contribution_profit / net_revenue if net_revenue > 0 else 0
    
    customer_ltv = lifetime_orders * max(contribution_profit, 1)
    
    rows.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_date": order_date.date(),
        "region": region,
        "city": city,
        "customer_segment": segment,
        "acquisition_channel": acquisition_channel,
        "new_vs_repeat": new_vs_repeat,
        "subscription_member": subscription_member,
        "retention_score": round(retention_score, 3),
        "lifetime_orders": lifetime_orders,
        "customer_ltv": round(customer_ltv, 2),
        "restaurant_category": restaurant_category,
        "gross_order_value": round(gross_order_value, 2),
        "discount_amount": round(discount_amount, 2),
        "discount_pct": round(discount_pct * 100, 2),
        "delivery_fee": round(delivery_fee, 2),
        "service_fee": round(service_fee, 2),
        "refund_amount": round(refund_amount, 2),
        "refund_reason": refund_reason,
        "refund_rate": round((refund_amount / gross_order_value) * 100, 2),
        "delivery_distance_miles": round(delivery_distance_miles, 2),
        "delivery_time_minutes": round(delivery_time_minutes, 1),
        "driver_incentive_cost": round(driver_incentive_cost, 2),
        "marketing_cac_allocated": round(marketing_cac_allocated, 2),
        "restaurant_commission": round(restaurant_commission, 2),
        "gross_margin": round(gross_margin, 2),
        "net_revenue": round(net_revenue, 2),
        "contribution_profit": round(contribution_profit, 2),
        "contribution_margin_pct": round(contribution_margin_pct * 100, 2),
        "promotion_type": promotion_type
    })

df = pd.DataFrame(rows)

display(df.head())
print(df.shape)


In [0]:
# Save as Databricks table

spark_df = spark.createDataFrame(df)

spark_df.write.mode("overwrite").saveAsTable("doordash_margin_orders")

print("Table created: doordash_margin_orders")

In [0]:
%sql
-- QA Check

SELECT
  COUNT(*) AS total_orders,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue,
  ROUND(SUM(contribution_profit), 2) AS total_contribution_profit,
  ROUND(SUM(contribution_profit) / SUM(net_revenue) * 100, 2) AS contribution_margin_pct
FROM doordash_margin_orders;

In [0]:
%sql
SELECT
  customer_segment,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(SUM(contribution_profit), 2) AS contribution_profit,
  ROUND(SUM(contribution_profit) / SUM(net_revenue) * 100, 2) AS contribution_margin_pct,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  ROUND(AVG(retention_score), 2) AS avg_retention_score
FROM doordash_margin_orders
GROUP BY customer_segment
ORDER BY net_revenue DESC;

In [0]:
display(spark.table("doordash_margin_orders"))